# Phase 2: Fine-Tuning Llama 3 8B with Unsloth (Colab Edition)

This notebook takes the synthetic legal contract dataset generated locally and uses it to fine-tune a quantization-ready Small Language Model (SLM). We use Unsloth for highly optimized memory usage and faster training on a free Colab T4 GPU.

### Step 1: Install Dependencies
We use `%pip` to install the official Colab-optimized version of Unsloth and its dependencies.

In [ ]:
%%capture
%pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
%pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes datasets

### Step 2: Load the Base Model
We load Llama-3.2 3B in 4-bit precision so it fits easily within the 16GB VRAM of a free Google Colab T4 GPU.

In [ ]:
import os
import torch
from unsloth import FastLanguageModel

max_seq_length = 2048 # Legal contracts can be long, so we allow up to 2048 tokens.
dtype = None # Auto-detects float16 or bfloat16 based on your hardware
load_in_4bit = True # CRITICAL for memory savings

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct", # Using the 3B model we tested earlier
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

### Step 3: Apply LoRA Adapters
Instead of updating all 3 billion parameters, we use LoRA to inject small, trainable "adapter" matrices. We only train about 1% of the total model.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, 
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Optimized to 0 for Unsloth
    bias = "none",    # Optimized to "none" for Unsloth
    use_gradient_checkpointing = "unsloth", # Saves massive amounts of VRAM
    random_state = 3407,
    use_rslora = False,
)

### Step 4: Load and Format the Synthetic Dataset
**⚠️ IMPORTANT:** Make sure you have uploaded `training_dataset.jsonl` to the files panel on the left before running this cell!

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

dataset_path = "training_dataset.jsonl"

if not os.path.exists(dataset_path):
    print(f"❌ Error: Could not find dataset at '{dataset_path}'")
    print("Please click the Folder icon on the left and upload your JSONL file!")
else:
    dataset = load_dataset("json", data_files=dataset_path, split="train")

    tokenizer = get_chat_template(
        tokenizer,
        chat_template = "llama-3",
        mapping = {"role" : "role", "content" : "content", "user" : "user", "assistant" : "assistant"}
    )

    def formatting_prompts_func(examples):
        convos = examples["messages"]
        texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
        return { "text" : texts }

    dataset = dataset.map(formatting_prompts_func, batched = True)
    print(f"✅ Successfully loaded and formatted {len(dataset)} examples ready for training.")

### Step 5: Start Training
This will launch the training loop. On a free Colab GPU, it should take roughly 10-15 minutes to chew through the dataset.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 100, # Adjust higher if you have more than 500 examples
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

### Step 6: Save the Trained LoRA Adapters
Once training is done, we save the custom LoRA weights. **Download this folder to your PC** when it finishes!

In [ ]:
model.save_pretrained("legal_slm_lora_adapter")
tokenizer.save_pretrained("legal_slm_lora_adapter")
print("✅ Fine-Tuned Adapters saved successfully! Check the file menu on the left to download 'legal_slm_lora_adapter'.")